# Apps Migration: Inventory Generation

This notebook discovers all Databricks Apps in the source workspace and generates an inventory CSV for review.

## Configuration

In [ ]:
try:
    catalog = dbutils.widgets.get("catalog")
except:
    catalog = "YOUR_SOURCE_CATALOG"

try:
    schema = dbutils.widgets.get("schema")
except:
    schema = "apps_demo"

try:
    volume = dbutils.widgets.get("volume")
except:
    volume = "apps_migration"

volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
inventory_path = f"{volume_path}/apps_inventory"

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Volume: {volume}")
print(f"Inventory path: {inventory_path}")

## Initialize Workspace Client

In [ ]:
from databricks.sdk import WorkspaceClient
from datetime import datetime
import requests

w = WorkspaceClient()
workspace_url = w.config.host
print(f"Connected to: {workspace_url}")

## Discover all Apps in workspace

In [ ]:
print("Discovering apps...")

# Use REST API to list apps
api_url = f"{workspace_url.rstrip('/')}/api/2.0/apps"
headers = w.config.authenticate()
response = requests.get(api_url, headers=headers)
response.raise_for_status()

apps_data = response.json().get("apps", [])

apps = []
for app in apps_data:
    source_path = ""
    status = "UNKNOWN"
    
    if app.get('active_deployment'):
        source_path = app['active_deployment'].get('source_code_path', '')
        status = app['active_deployment'].get('status', {}).get('state', 'UNKNOWN')
    
    apps.append({
        "name": app.get('name', ''),
        "description": app.get('description', ''),
        "url": app.get('url', ''),
        "status": status,
        "source_code_path": source_path,
        "creator": app.get('creator', ''),
        "create_time": app.get('create_time', ''),
        "env_var_count": 0,
    })

print(f"Found {len(apps)} apps")
for app in apps:
    print(f"  - {app['name']}: {app['status']}")

## Write inventory CSV to volume

In [ ]:
import csv
from io import StringIO, BytesIO

if not apps:
    print("No apps found. Nothing to export.")
else:
    fieldnames = ["name", "description", "url", "status", "source_code_path", "creator", "create_time", "env_var_count"]
    
    output = StringIO()
    writer = csv.DictWriter(output, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(apps)
    
    csv_path = f"{inventory_path}/apps_inventory.csv"
    w.files.upload(csv_path, BytesIO(output.getvalue().encode('utf-8')), overwrite=True)
    
    print(f"Inventory written to: {csv_path}")

## Display summary

In [ ]:
import pandas as pd

if apps:
    df = pd.DataFrame(apps)
    
    print("=" * 60)
    print("INVENTORY SUMMARY")
    print("=" * 60)
    print(f"Total apps discovered: {len(df)}")
    print(f"Inventory file: {csv_path}")
    print("=" * 60)
    
    display(df)
else:
    print("No apps discovered.")